# Tutorial 2 - The Leaky Integrate-and-Fire Neuron
Tutorial code and runnable examples from the snnTorch tutorial (Lapicque LIF).

## Setup
If you don't have `snntorch` installed in your environment, run:
```
pip install snntorch
```

The cells below contain the imports and helper plotting functions used in the tutorial.

In [ ]:
# Imports
import snntorch as snn
from snntorch import spikeplot as splt
from snntorch import spikegen

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

print('snntorch', snn.__version__ if hasattr(snn, '__version__') else 'unknown')
print('torch', torch.__version__)

## Simple Forward-Euler Leaky Integrator (from first principles)
This cell implements a basic forward-Euler solver for the passive membrane ODE and plots the decay.

In [ ]:
def leaky_integrate_neuron(U, time_step=1e-3, I=0.0, R=5.0, C=1e-3):
    tau = R * C
    U = U + (time_step / tau) * (-U + I * R)
    return U

def plot_mem_trace(U_trace, title='Leaky Neuron'):
    arr = np.asarray(U_trace)
    plt.figure(figsize=(8, 4))
    plt.plot(arr, marker='o')
    plt.title(title)
    plt.xlabel('Time step')
    plt.ylabel('Membrane potential U')
    plt.grid(True)
    plt.show()

# Run a short simulation: decay from U0=0.9 with no input
num_steps = 100
U = 0.9
U_trace = []
for step in range(num_steps):
    U_trace.append(U)
    U = leaky_integrate_neuron(U)

plot_mem_trace(U_trace, 'Passive membrane decay (forward Euler)')

## Lapicque LIF using snnTorch
Now we use `snn.Lapicque` to reproduce the tutorial examples: no stimulus, step input, pulse inputs, and spiking with reset. Helper plotting utilities follow.

In [ ]:
# Helper plotting functions used in the tutorial
def plot_current_pulse_response(current, memrec, title, vline1=None, vline2=None, ylim_max=None, out_path=None):
    cur_np = current.detach().cpu().numpy().squeeze() if isinstance(current, torch.Tensor) else np.asarray(current).squeeze()
    mem_np = memrec.detach().cpu().numpy().squeeze() if isinstance(memrec, torch.Tensor) else np.asarray(memrec).squeeze()

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    ax1.plot(cur_np, color='C0')
    ax1.set_ylabel('Input current')
    if vline1 is not None:
        ax1.axvline(vline1, color='k', linestyle='--')
    if vline2 is not None:
        ax1.axvline(vline2, color='k', linestyle='--')

    ax2.plot(mem_np, marker='o')
    ax2.axhline(y=1.0, color='r', linestyle='--', label='Threshold')
    ax2.set_xlabel('Time step')
    ax2.set_ylabel('Membrane potential (U)')
    ax2.legend()
    ax2.grid(True)

    fig.suptitle(title)
    fig.tight_layout()
    if out_path is not None:
        fig.savefig(out_path, dpi=200)
        print(f'Saved plot to {out_path}')
    plt.show()

def plot_cur_mem_spk(cur_in, mem_rec, spk_rec, thr_line=1.0, vline=None, ylim_max2=None, title=None):
    cur_np = cur_in.detach().cpu().numpy().squeeze() if isinstance(cur_in, torch.Tensor) else np.asarray(cur_in).squeeze()
    mem_np = mem_rec.detach().cpu().numpy().squeeze() if isinstance(mem_rec, torch.Tensor) else np.asarray(mem_rec).squeeze()
    spk_np = spk_rec.detach().cpu().numpy().squeeze() if isinstance(spk_rec, torch.Tensor) else np.asarray(spk_rec).squeeze()

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    ax1.plot(cur_np, color='C0')
    if vline is not None:
        ax1.axvline(vline, color='k', linestyle='--')
    ax1.set_ylabel('Input current')

    ax2.plot(mem_np, label='Membrane')
    ax2.plot(spk_np * (mem_np.max() if mem_np.size else 1.0), 'k|', label='Spikes')
    ax2.axhline(y=thr_line, color='r', linestyle='--', label='Threshold')
    ax2.set_xlabel('Time step')
    ax2.set_ylabel('Membrane / Spikes')
    if ylim_max2 is not None:
        ax2.set_ylim(top=ylim_max2)
    ax2.legend()
    ax2.grid(True)

    if title is not None:
        fig.suptitle(title)
    fig.tight_layout()
    plt.show()

In [ ]:
# Lapicque examples
time_step = 1e-3
R = 5.0
C = 1e-3
lif1 = snn.Lapicque(R=R, C=C, time_step=time_step)

# 3.1 Lapicque without stimulus
num_steps = 100
mem = torch.ones(1) * 0.9
cur_in = torch.zeros(num_steps, 1)
spk_out = torch.zeros(1)
mem_rec = [mem]
for step in range(num_steps):
    spk_out, mem = lif1(cur_in[step], mem)
    mem_rec.append(mem)
mem_rec = torch.stack(mem_rec)
plot_current_pulse_response(cur_in, mem_rec, "Lapicque without stimulus")

### Step input and pulse examples
Run the following cells to reproduce the step/pulse examples from the tutorial.

In [ ]:
# Step input example
num_steps = 200
cur_in = torch.cat((torch.zeros(10, 1), torch.ones(190, 1) * 0.2), 0)
mem = torch.zeros(1)
spk_out = torch.zeros(1)
mem_rec = [mem]
spk_rec = [spk_out]
for step in range(num_steps):
    spk_out, mem = lif1(cur_in[step], mem)
    mem_rec.append(mem)
    spk_rec.append(spk_out)
mem_rec = torch.stack(mem_rec)
spk_rec = torch.stack(spk_rec)
plot_cur_mem_spk(cur_in, mem_rec, spk_rec, thr_line=1.0, vline=109, title="Lapicque step input")

In [ ]:
# Pulse input examples (three pulses of different widths/amplitudes)
num_steps = 200
# pulse 1: on at 10, off at 30
cur_in1 = torch.cat((torch.zeros(10, 1), torch.ones(20, 1) * 0.1, torch.zeros(170, 1)), 0)
mem = torch.zeros(1)
spk_out = torch.zeros(1)
mem_rec1 = [mem]
for step in range(num_steps):
    spk_out, mem = lif1(cur_in1[step], mem)
    mem_rec1.append(mem)
mem_rec1 = torch.stack(mem_rec1)
plot_current_pulse_response(cur_in1, mem_rec1, "Lapicque pulse 1", vline1=10, vline2=30)

# pulse 2: shorter, larger amplitude
cur_in2 = torch.cat((torch.zeros(10, 1), torch.ones(10, 1) * 0.111, torch.zeros(180, 1)), 0)
mem = torch.zeros(1)
mem_rec2 = [mem]
for step in range(num_steps):
    spk_out, mem = lif1(cur_in2[step], mem)
    mem_rec2.append(mem)
mem_rec2 = torch.stack(mem_rec2)
plot_current_pulse_response(cur_in2, mem_rec2, "Lapicque pulse 2", vline1=10, vline2=20)

# pulse 3: very short, higher amplitude
cur_in3 = torch.cat((torch.zeros(10, 1), torch.ones(5, 1) * 0.147, torch.zeros(185, 1)), 0)
mem = torch.zeros(1)
mem_rec3 = [mem]
for step in range(num_steps):
    spk_out, mem = lif1(cur_in3[step], mem)
    mem_rec3.append(mem)
mem_rec3 = torch.stack(mem_rec3)
plot_current_pulse_response(cur_in3, mem_rec3, "Lapicque pulse 3", vline1=10, vline2=15)

## Spiking with reset and comparisons
Demonstrate uncontrolled spiking (no reset) vs. LIF with reset and the built-in Lapicque neuron.

In [ ]:
# Custom LIF with reset (simple implementation)
def leaky_integrate_and_fire(mem, cur=0.0, threshold=1.0, time_step=1e-3, R=5.1, C=5e-3):
    tau_mem = R * C
    spk = (mem > threshold).float() if isinstance(mem, torch.Tensor) else float(mem > threshold)
    mem = mem + (time_step / tau_mem) * (-mem + cur * R) - spk * threshold
    return mem, spk

# uncontrolled spiking (no reset)
num_steps = 200
cur_in = torch.cat((torch.zeros(10), torch.ones(190) * 0.2), 0)
mem = torch.zeros(1)
mem_rec = []
spk_rec = []
for step in range(num_steps):
    mem, spk = leaky_integrate_and_fire(mem, cur_in[step])
    mem_rec.append(mem)
    spk_rec.append(spk)
mem_rec = torch.stack(mem_rec)
spk_rec = torch.stack(spk_rec)
plot_cur_mem_spk(cur_in, mem_rec, spk_rec, thr_line=1.0, title="Uncontrolled spiking (no reset)")

# LIF (with reset) using same function but threshold enforced in reset step above
mem = torch.zeros(1)
mem_rec = []
spk_rec = []
for step in range(num_steps):
    mem, spk = leaky_integrate_and_fire(mem, cur_in[step])
    mem_rec.append(mem)
    spk_rec.append(spk)
mem_rec = torch.stack(mem_rec)
spk_rec = torch.stack(spk_rec)
plot_cur_mem_spk(cur_in, mem_rec, spk_rec, thr_line=1.0, title="LIF with reset (simple)")